# ZA-GAS model comparison: phi-only vs phi-xi

This notebook runs the comparison requested for the thesis workflow:

- `phi` only time-varying, with `xi` static.
- `phi` and `xi` both time-varying.
- In-sample PIT fitting and quantile-residual ACF.
- 95% confidence intervals for estimated hyper-parameters.
- Kupiec and Christoffersen 95% coverage tests for IS, OOS, and full sample.
- IS/OOS CRPS, log likelihood, AIC, BIC, RMSE, and MAD.
- OOS expected value and quantiles: 50%, 75%, 90%, 95%, 97%, 99%, simulated min, simulated max.
- CSV cache files for costly results, plus PDF and LaTeX report output.

The CSV and `.tex` cache outputs are ignored by git. The PDF is intentionally not globally ignored.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from models.factory import build_zagas_model
from models.parameters import load_theta_csv
from diagnostics.residuals import quantile_residuals, pit_values
from diagnostics.tests import jarque_bera
from report.workflow import evaluate_and_cache_model
from report.summarize import display_model_label, render_model_report

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
print('Imports ok')

Imports ok


## Configuration

Use `RUN_SCOPE` to decide what is estimated or loaded in this notebook:

- `RUN_SCOPE = 'monthly'`: run/load only the monthly models. Monthly estimation uses the thesis trial specification: standard `BFGS` with `use_bounds=False`, saved under `MONTHLY_BFGS_CACHE_DIR`.
- `RUN_SCOPE = 'daily'`: run/load only the daily models, using the existing daily cache and leaving monthly results out of the report.
- `RUN_SCOPE = 'both'`: load daily models from cache, run/load monthly models, and create the combined report.

Set `N_MAX_LOCATIONS = None` to run every train/test pair. During development, keep it at `1` so you do not accidentally launch all costly fits.

Current data folders:

- Daily: `C:/Users/ilang/OneDrive/Documentos/Ilan/academia/dissertação/data/output`
- Monthly: `C:/Users/ilang/OneDrive/Documentos/Ilan/academia/dissertação/data/monthly output`

To regenerate the report without re-estimating, keep `FORCE_REFIT = False`, keep the same cache directories, and rerun the executable cells from **Configuration** through **Render PDF and LaTeX report**. Cached `estimated_parameters.csv` files are loaded instead of optimizing again. Daily refits are blocked unless you explicitly set `ALLOW_DAILY_REFIT = True`. When you change `MONTHLY_DIR` to a different dataset, also use a new `MONTHLY_BFGS_CACHE_DIR` or set `FORCE_REFIT = True` intentionally.

In [2]:
DAILY_DIR = Path(r'C:/Users/ilang/OneDrive/Documentos/Ilan/academia/dissertação/data/output')
MONTHLY_DIR = Path(r'C:/Users/ilang/OneDrive/Documentos/Ilan/academia/dissertação/data/monthly output')

Y_COL = 'PRECIPITACAO TOTAL, DIARIO(mm)'
DATE_COL = 'Data Medicao'

N_MAX_LOCATIONS = 1
RUN_SCOPE = 'both'  # choose: 'monthly', 'daily', or 'both'
if RUN_SCOPE not in {'monthly', 'daily', 'both'}:
    raise ValueError("RUN_SCOPE must be one of: 'monthly', 'daily', 'both'")
RUN_DAILY = RUN_SCOPE in {'daily', 'both'}
RUN_MONTHLY = RUN_SCOPE in {'monthly', 'both'}

MODEL_TYPES = ['phi', 'phi_xi']
FORCE_REFIT = False
N_DRAWS = 500
SEED = 42

CACHE_DIR = Path('artifacts/cache')
MONTHLY_BFGS_CACHE_DIR = Path('artifacts/cache_monthly_output_bfgs_216')
ALLOW_DAILY_REFIT = False
MONTHLY_FIT_METHOD = 'BFGS'
MONTHLY_USE_BOUNDS = False
USE_MONTHLY_BFGS_TRIAL = True
REPORT_DIR = Path('artifacts/reports')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MONTHLY_BFGS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def load_pairs(directory: Path, y_col: str, date_col: str, n_max=None) -> dict:
    train_files = sorted(f for f in directory.iterdir() if f.name.endswith('_train.csv'))
    if n_max is not None:
        train_files = train_files[:n_max]

    out = {}
    for train_file in train_files:
        loc = train_file.stem.replace('_train', '')
        test_file = train_file.parent / f'{loc}_test.csv'
        if not test_file.exists():
            print(f'WARNING: no test file for {loc}; skipping')
            continue

        df_train = pd.read_csv(train_file)
        df_test = pd.read_csv(test_file)
        out[loc] = {
            'y_train': df_train[y_col].to_numpy(dtype=float),
            'y_test': df_test[y_col].to_numpy(dtype=float),
            'date_train': df_train[date_col].to_numpy() if date_col in df_train.columns else None,
            'date_test': df_test[date_col].to_numpy() if date_col in df_test.columns else None,
            'train_file': train_file,
            'test_file': test_file,
        }
    return out

daily_data = load_pairs(DAILY_DIR, Y_COL, DATE_COL, N_MAX_LOCATIONS) if RUN_DAILY else {}
monthly_data = load_pairs(MONTHLY_DIR, Y_COL, DATE_COL, N_MAX_LOCATIONS) if RUN_MONTHLY else {}

print('Daily locations:', list(daily_data))
print('Monthly locations:', list(monthly_data))

Daily locations: ['BELO HORIZONTE']
Monthly locations: ['BELO HORIZONTE']


## Parameter-count sanity check

For daily `phi_xi`, the expected total is 43: two time-varying positive-part parameters, seven GAS lags, two static GB2 parameters, and nine pi-dynamics parameters.

In [4]:
for seasonal in [s for s in ['monthly', 'daily'] if (s == 'monthly' and RUN_MONTHLY) or (s == 'daily' and RUN_DAILY)]:
    for model_type in MODEL_TYPES:
        model = build_zagas_model(model_type, seasonal=seasonal)
        print(model_type, seasonal, model.parameter_count_breakdown())

phi monthly {'seasonal': 'monthly', 'lags': [1, 2, 3, 11, 12, 13], 'n_lags': 6, 'tv_parameters': ['phi'], 'per_tv_parameter': 14, 'gas_dynamic_parameters': 14, 'static_positive_parameters': 3, 'pi_parameters': 8, 'total_parameters': 25, 'formula': 'n_tv * (omega + f0 + A_lags + B_lags) + n_static_positive + n_pi'}
phi_xi monthly {'seasonal': 'monthly', 'lags': [1, 2, 3, 11, 12, 13], 'n_lags': 6, 'tv_parameters': ['phi', 'xi'], 'per_tv_parameter': 14, 'gas_dynamic_parameters': 28, 'static_positive_parameters': 2, 'pi_parameters': 8, 'total_parameters': 38, 'formula': 'n_tv * (omega + f0 + A_lags + B_lags) + n_static_positive + n_pi'}
phi daily {'seasonal': 'daily', 'lags': [1, 2, 3, 364, 365, 366, 367], 'n_lags': 7, 'tv_parameters': ['phi'], 'per_tv_parameter': 16, 'gas_dynamic_parameters': 16, 'static_positive_parameters': 3, 'pi_parameters': 9, 'total_parameters': 28, 'formula': 'n_tv * (omega + f0 + A_lags + B_lags) + n_static_positive + n_pi'}
phi_xi daily {'seasonal': 'daily', 'lag

## Fit or load models, then cache diagnostics

If an `estimated_parameters.csv` already exists and `FORCE_REFIT = False`, the notebook reloads `theta` from CSV instead of re-estimating. If the rest of the cached diagnostic CSVs exist, those are loaded too. Otherwise, diagnostics are recomputed from the loaded parameters.

For monthly models, this notebook uses the BFGS/no-bounds specification configured above:

```python
MONTHLY_FIT_METHOD = 'BFGS'
MONTHLY_USE_BOUNDS = False
MONTHLY_BFGS_CACHE_DIR = Path('artifacts/cache_monthly_output_bfgs_216')
```

To run only monthly models on the larger monthly dataset, set `RUN_SCOPE = 'monthly'`, confirm `MONTHLY_DIR` points to the desired folder, and run the notebook from **Configuration** through this section. To keep the daily models in the same report while updating only monthly results, set `RUN_SCOPE = 'both'`; daily will load from cache when `FORCE_REFIT = False`. To regenerate the report from cached results, keep `FORCE_REFIT = False` and rerun the cells through **Render PDF and LaTeX report**.

In [5]:
def safe_model_id(seasonal: str, loc: str, model_type: str) -> str:
    clean_loc = ''.join(ch if ch.isalnum() else '_' for ch in loc).strip('_').lower()
    return f'{seasonal}_{clean_loc}_{model_type}'


def series_id(model_id: str) -> str:
    if model_id.endswith('_phi_xi'):
        return model_id[:-7]
    if model_id.endswith('_phi'):
        return model_id[:-4]
    return model_id


def make_series_summary(model_id: str, data: dict) -> pd.DataFrame:
    rows = []
    for sample, values in [
        ('IS', np.asarray(data['y_train'], dtype=float)),
        ('OOS', np.asarray(data['y_test'], dtype=float)),
        ('Full', np.concatenate([np.asarray(data['y_train'], dtype=float), np.asarray(data['y_test'], dtype=float)])),
    ]:
        x = values[np.isfinite(values)]
        qs = np.quantile(x, [0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
        rows.append({
            'series_id': series_id(model_id),
            'model_id': model_id,
            'sample': sample,
            'n': len(x),
            'min': np.min(x),
            'max': np.max(x),
            'mean': np.mean(x),
            'std': np.std(x, ddof=1),
            'prop_zero': np.mean(x == 0.0),
            'q01': qs[0], 'q05': qs[1], 'q25': qs[2], 'q50': qs[3],
            'q75': qs[4], 'q90': qs[5], 'q95': qs[6], 'q99': qs[7],
        })
    return pd.DataFrame(rows)


def make_series_values(model_id: str, data: dict) -> pd.DataFrame:
    return pd.concat([
        pd.DataFrame({
            'series_id': series_id(model_id),
            'model_id': model_id,
            'sample': 'IS',
            't_index': np.arange(len(data['y_train'])),
            'y': np.asarray(data['y_train'], dtype=float),
        }),
        pd.DataFrame({
            'series_id': series_id(model_id),
            'model_id': model_id,
            'sample': 'OOS',
            't_index': np.arange(len(data['y_test'])),
            'y': np.asarray(data['y_test'], dtype=float),
        }),
    ], ignore_index=True)


def make_is_diagnostic_series(model, theta, model_id: str, data: dict) -> pd.DataFrame:
    rng = np.random.default_rng(SEED)
    paths = model.filter(theta, data['y_train'])
    cdfs = model.cdf_series(theta, data['y_train'])
    y_eff = paths['y_eff']
    is_pit = pit_values(cdfs, y_eff, randomise_zeros=True, rng=rng)
    is_qr = quantile_residuals(cdfs, y_eff, randomise_zeros=True, rng=rng)
    return pd.DataFrame({
        'model_id': model_id,
        'sample': 'IS',
        't_index': np.arange(len(is_pit)),
        'pit': is_pit,
        'quantile_residual': is_qr,
    })


def make_jb_from_diagnostic_series(model_id: str, diagnostic_series: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for sample in ['IS']:
        qr = diagnostic_series.loc[
            (diagnostic_series['model_id'] == model_id) & (diagnostic_series['sample'] == sample),
            'quantile_residual'
        ].dropna().to_numpy()
        d = jarque_bera(qr)
        rows.append({
            'model_id': model_id,
            'sample': sample,
            'n': d['n'],
            'skewness': d['skewness'],
            'kurtosis': d['kurtosis'],
            'JB_stat': d['stat'],
            'pvalue': d['pvalue'],
            'reject_5pct': d['pvalue'] < 0.05,
        })
    return pd.DataFrame(rows)


def cache_base_for_model(seasonal: str, model_id: str) -> Path:
    if seasonal == 'monthly' and USE_MONTHLY_BFGS_TRIAL:
        return MONTHLY_BFGS_CACHE_DIR / model_id
    return CACHE_DIR / model_id


def cache_output_dir_for_seasonal(seasonal: str) -> Path:
    if seasonal == 'monthly' and USE_MONTHLY_BFGS_TRIAL:
        return MONTHLY_BFGS_CACHE_DIR
    return CACHE_DIR


def cached_result(model, model_id: str, data: dict):
    seasonal = 'monthly' if model_id.startswith('monthly_') else 'daily'
    base = cache_base_for_model(seasonal, model_id)
    required = {
        'parameters': base / 'estimated_parameters.csv',
        'metrics': base / 'metrics.csv',
        'oos_forecasts': base / 'oos_forecasts.csv',
        'acf': base / 'quantile_residual_acf.csv',
        'pit': base / 'pit_fit.csv',
        'coverage': base / 'coverage_95.csv',
    }
    if not all(path.exists() for path in required.values()):
        return None

    result = {
        'model_id': model_id,
        'model': model,
        'parameters': pd.read_csv(required['parameters']),
        'metrics_frame': pd.read_csv(required['metrics']),
        'oos_forecasts': pd.read_csv(required['oos_forecasts']),
        'acf': pd.read_csv(required['acf']),
        'pit': pd.read_csv(required['pit']),
        'coverage': pd.read_csv(required['coverage']),
        'csv': required.copy(),
        'description': 'Loaded from cached CSV files.',
    }

    optional = {
        'diagnostic_series': base / 'diagnostic_series.csv',
        'jarque_bera': base / 'jarque_bera.csv',
        'series_summary': base / 'series_summary.csv',
        'series_values': base / 'series_values.csv',
    }

    for key, path in optional.items():
        if path.exists():
            result[key] = pd.read_csv(path)
            result['csv'][key] = path

    # Report-only additions. These never call model.fit().
    if 'series_summary' not in result:
        result['series_summary'] = make_series_summary(model_id, data)
        optional['series_summary'].parent.mkdir(parents=True, exist_ok=True)
        result['series_summary'].to_csv(optional['series_summary'], index=False)
        result['csv']['series_summary'] = optional['series_summary']

    if 'series_values' not in result:
        result['series_values'] = make_series_values(model_id, data)
        optional['series_values'].parent.mkdir(parents=True, exist_ok=True)
        result['series_values'].to_csv(optional['series_values'], index=False)
        result['csv']['series_values'] = optional['series_values']

    if 'diagnostic_series' not in result:
        theta = load_theta_csv(required['parameters'], model_id=model_id)
        result['diagnostic_series'] = make_is_diagnostic_series(model, theta, model_id, data)
        result['diagnostic_series'].to_csv(optional['diagnostic_series'], index=False)
        result['csv']['diagnostic_series'] = optional['diagnostic_series']

    if 'jarque_bera' not in result:
        result['jarque_bera'] = make_jb_from_diagnostic_series(model_id, result['diagnostic_series'])
        result['jarque_bera'].to_csv(optional['jarque_bera'], index=False)
        result['csv']['jarque_bera'] = optional['jarque_bera']

    return result


def fit_or_load_then_cache(seasonal: str, loc: str, data: dict, model_type: str):
    model_id = safe_model_id(seasonal, loc, model_type)
    model = build_zagas_model(model_type, seasonal=seasonal)

    if not FORCE_REFIT:
        cached = cached_result(model, model_id, data)
        if cached is not None:
            print(f'Loaded cached diagnostics: {model_id}')
            return cached

    output_dir = cache_output_dir_for_seasonal(seasonal)
    param_csv = output_dir / model_id / 'estimated_parameters.csv'
    if param_csv.exists() and not FORCE_REFIT:
        theta = load_theta_csv(param_csv, model_id=model_id)
        fit = {
            'theta': theta,
            'loglik': model.loglik(theta, data['y_train']),
            'success': True,
            'result': None,
        }
        print(f'Loaded theta, recomputing missing diagnostics: {model_id}')
    else:
        if seasonal == 'daily' and not ALLOW_DAILY_REFIT:
            raise RuntimeError(f'Missing daily cache for {model_id}. Daily refit is disabled by ALLOW_DAILY_REFIT=False.')
        print(f'Fitting {model_id}  n_train={len(data["y_train"]):,}  n_test={len(data["y_test"]):,}')
        if seasonal == 'monthly' and USE_MONTHLY_BFGS_TRIAL:
            print(f'  monthly optimizer: method={MONTHLY_FIT_METHOD}, use_bounds={MONTHLY_USE_BOUNDS}')
            fit = model.fit(data['y_train'], verbose=False, method=MONTHLY_FIT_METHOD, use_bounds=MONTHLY_USE_BOUNDS)
        else:
            fit = model.fit(data['y_train'], verbose=False)
        print(f'  loglik={fit["loglik"]:.3f} success={fit["success"]}')

    result = evaluate_and_cache_model(
        model=model,
        fit=fit,
        y_train=data['y_train'],
        y_test=data['y_test'],
        model_id=model_id,
        output_dir=output_dir,
        n_draws=N_DRAWS,
        seed=SEED,
    )
    result['description'] = f'{seasonal} {loc}: {model_type}'
    return result


In [6]:
all_results = []

for seasonal, dataset in [('daily', daily_data), ('monthly', monthly_data)]:
    for loc, data in dataset.items():
        for model_type in MODEL_TYPES:
            all_results.append(fit_or_load_then_cache(seasonal, loc, data, model_type))

print(f'Collected {len(all_results)} model result objects')

Loaded cached diagnostics: daily_belo_horizonte_phi
Loaded cached diagnostics: daily_belo_horizonte_phi_xi
Fitting monthly_belo_horizonte_phi  n_train=216  n_test=24
  monthly optimizer: method=BFGS, use_bounds=False


  loglik=-1053.809 success=False


Fitting monthly_belo_horizonte_phi_xi  n_train=216  n_test=24
  monthly optimizer: method=BFGS, use_bounds=False


C:\Users\ilang\OneDrive\Documentos\Ilan\academia\dissertação\python attempt\FurtherTopics\distributions\gb2_log_link.py:52: RuntimeWarning: divide by zero encountered in log
  - np.log(beta_fn(a, b))


  loglik=-1017.678 success=False


Collected 4 model result objects


## Comparison tables

In [7]:
metrics = pd.concat([r['metrics_frame'] for r in all_results], ignore_index=True)
coverage = pd.concat([r['coverage'] for r in all_results], ignore_index=True)
acf = pd.concat([r['acf'] for r in all_results], ignore_index=True)
pit = pd.concat([r['pit'] for r in all_results], ignore_index=True)
params = pd.concat([r['parameters'] for r in all_results], ignore_index=True)
jarque_bera_tables = pd.concat([r['jarque_bera'] for r in all_results if 'jarque_bera' in r], ignore_index=True)
series_summary = pd.concat([r['series_summary'] for r in all_results if 'series_summary' in r], ignore_index=True)

def readable(df):
    out = df.copy()
    if 'model_id' in out.columns:
        out.insert(0, 'Model', out['model_id'].map(display_model_label))
    return out

display(readable(metrics))
display(readable(coverage))
display(readable(acf).pivot_table(index=['Model', 'sample'], columns='lag', values='acf'))
display(readable(pit))
display(readable(params).head(20))
display(readable(jarque_bera_tables))
display(series_summary.drop_duplicates(subset=['series_id', 'sample']))

,Model,model_id,sample,loglik,aic,bic,rmse,mad,crps
0,Daily $\phi$-only,daily_belo_horizonte_phi,IS,-5035.183572,10126.367143,10297.086557,12.085465,7.513113,2.678016
1,Daily $\phi$-only,daily_belo_horizonte_phi,OOS,-1120.375652,2296.751304,2425.394881,11.638355,7.575449,2.619942
2,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,IS,-5017.619165,10121.238329,10383.414572,13.313079,8.921411,2.667160
3,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,OOS,-1130.566470,2347.132940,2544.692718,13.669480,9.161822,2.631246
4,Monthly $\phi$-only,monthly_belo_horizonte_phi,IS,-1053.809389,2157.618777,2240.448927,197.350278,129.046156,29.047085
5,Monthly $\phi$-only,monthly_belo_horizonte_phi,OOS,-119.431565,288.863130,318.314476,183.040715,126.160475,26.752770
6,"Monthly $\phi,\xi$",monthly_belo_horizonte_phi_xi,IS,-1017.678481,2111.356963,2237.258790,226.185402,160.740526,22.985969
7,"Monthly $\phi,\xi$",monthly_belo_horizonte_phi_xi,OOS,-155.305491,386.610982,431.377027,579.004715,312.083366,92.704298


,Model,model_id,sample,coverage,alpha,violations,expected_violations,violation_rate,LR_uc,pvalue_uc,LR_ind,LR_cc,pvalue_cc,reject_uc_5pct,reject_cc_5pct
0,Daily $\phi$-only,daily_belo_horizonte_phi,IS,95%,0.05,139,164.25,0.042314,4.301178,0.038086,1.568757,5.869935,0.053132,True,False
1,Daily $\phi$-only,daily_belo_horizonte_phi,OOS,95%,0.05,34,36.55,0.046512,0.191547,0.661632,0.269169,0.460716,0.794249,False,False
2,Daily $\phi$-only,daily_belo_horizonte_phi,Full,95%,0.05,173,200.80,0.043078,4.241922,0.039437,0.864944,5.106866,0.077814,True,False
3,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,IS,95%,0.05,143,164.25,0.043531,3.020492,0.082219,6.289172,9.309664,0.009516,False,True
4,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,OOS,95%,0.05,32,36.55,0.043776,0.621248,0.430584,0.139424,0.760672,0.683632,False,False
5,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,Full,95%,0.05,175,200.80,0.043576,3.640884,0.056377,4.761633,8.402517,0.014977,False,True
6,Monthly $\phi$-only,monthly_belo_horizonte_phi,IS,95%,0.05,6,10.15,0.029557,2.080101,0.149230,0.367404,2.447505,0.294124,False,False
7,Monthly $\phi$-only,monthly_belo_horizonte_phi,OOS,95%,0.05,4,1.20,0.166667,4.390652,0.036136,0.180848,4.571500,0.101698,True,False
8,Monthly $\phi$-only,monthly_belo_horizonte_phi,Full,95%,0.05,10,11.35,0.044053,0.175781,0.675025,0.583537,0.759318,0.684095,False,False
9,"Monthly $\phi,\xi$",monthly_belo_horizonte_phi_xi,IS,95%,0.05,11,10.15,0.054187,0.073026,0.786980,1.267717,1.340743,0.511518,False,False


lag                              1         2         3         11        12        13        364       365       366       367
Model               sample                                                                                                    
Daily $\phi$-only   IS      0.129572  0.053856  0.022629       NaN       NaN       NaN  0.036982  0.026484 -0.009269 -0.021535
                    OOS     0.120627  0.070091  0.038656       NaN       NaN       NaN  0.026473  0.047398  0.016462  0.041206
Daily $\phi,\xi$    IS      0.109228  0.051018  0.019395       NaN       NaN       NaN  0.022324  0.049628  0.011627 -0.017178
                    OOS     0.126829  0.048105  0.064446       NaN       NaN       NaN  0.024644  0.024031  0.016456  0.040499
Monthly $\phi$-only IS      0.222893  0.000231 -0.025438  0.230298  0.143322  0.219312       NaN       NaN       NaN       NaN
                    OOS     0.148371 -0.071760 -0.163552 -0.096863  0.219392  0.055860       NaN       NaN       NaN       NaN
Monthly $\phi,\xi$  IS      0.097988 -0.020334  0.040015  0.009395  0.085190 -0.027235       NaN       NaN       NaN       NaN
                    OOS     0.013557  0.113579  0.285140 -0.119399 -0.106515 -0.136168       NaN       NaN       NaN       NaN

,Model,n,ks_stat,pvalue,reject_5pct,model_id,sample
0,Daily $\phi$-only,3285,0.016519,0.327847,False,daily_belo_horizonte_phi,IS
1,Daily $\phi$-only,731,0.024907,0.745127,False,daily_belo_horizonte_phi,OOS
2,Daily $\phi$-only,4016,0.013247,0.477391,False,daily_belo_horizonte_phi,Full
3,"Daily $\phi,\xi$",3285,0.020344,0.130045,False,daily_belo_horizonte_phi_xi,IS
4,"Daily $\phi,\xi$",731,0.024002,0.784239,False,daily_belo_horizonte_phi_xi,OOS
5,"Daily $\phi,\xi$",4016,0.015615,0.278463,False,daily_belo_horizonte_phi_xi,Full
6,Monthly $\phi$-only,203,0.096538,0.042475,True,monthly_belo_horizonte_phi,IS
7,Monthly $\phi$-only,24,0.137988,0.700092,False,monthly_belo_horizonte_phi,OOS
8,Monthly $\phi$-only,227,0.098284,0.023209,True,monthly_belo_horizonte_phi,Full
9,"Monthly $\phi,\xi$",203,0.070372,0.255014,False,monthly_belo_horizonte_phi_xi,IS


,Model,model_id,sample,seasonal,model_tv_params,parameter_order,parameter,block,estimate,std_error,ci_lower_95,ci_upper_95,loglik,success
0,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,0,omega_phi,gas,0.752018,15.351805,-29.336967,30.841003,-5035.183572,False
1,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,1,f0_phi,gas,2.968461,10.317157,-17.252794,23.189717,-5035.183572,False
2,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,2,A_phi_1,gas,0.146720,1.683137,-3.152167,3.445608,-5035.183572,False
3,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,3,A_phi_2,gas,0.070494,3.429452,-6.651108,6.792096,-5035.183572,False
4,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,4,A_phi_3,gas,-0.090410,5.267271,-10.414071,10.233252,-5035.183572,False
5,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,5,A_phi_364,gas,-0.024425,1.295383,-2.563328,2.514478,-5035.183572,False
6,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,6,A_phi_365,gas,0.067999,3.296610,-6.393237,6.529235,-5035.183572,False
7,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,7,A_phi_366,gas,0.019832,2.306343,-4.500518,4.540182,-5035.183572,False
8,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,8,A_phi_367,gas,0.030683,1.332903,-2.581760,2.643126,-5035.183572,False
9,Daily $\phi$-only,daily_belo_horizonte_phi,IS,daily,phi,9,B_phi_1,gas,0.238575,13.144819,-25.524797,26.001946,-5035.183572,False


,Model,model_id,sample,n,skewness,kurtosis,JB_stat,pvalue,reject_5pct
0,Daily $\phi$-only,daily_belo_horizonte_phi,IS,3283,-0.062021,2.942903,2.550661,0.279339,False
1,"Daily $\phi,\xi$",daily_belo_horizonte_phi_xi,IS,3283,-0.086602,2.884823,5.918309,0.051863,False
2,Monthly $\phi$-only,monthly_belo_horizonte_phi,IS,203,-0.193101,2.978728,1.265399,0.531156,False
3,Monthly $\phi$-only,monthly_belo_horizonte_phi,OOS,24,-0.418352,2.667281,0.810777,0.666718,False
4,"Monthly $\phi,\xi$",monthly_belo_horizonte_phi_xi,IS,203,-0.246263,3.002630,2.051899,0.358456,False
5,"Monthly $\phi,\xi$",monthly_belo_horizonte_phi_xi,OOS,24,-0.946776,3.351092,3.708806,0.156546,False


,series_id,model_id,sample,n,min,max,mean,std,prop_zero,q01,q05,q25,q50,q75,q90,q95,q99
0,daily_belo_horizonte,daily_belo_horizonte_phi,IS,3650,0.0,171.8,4.059781,11.703045,0.720000,0.0,0.0,0.000,0.00,0.600,14.11,26.110,53.961
1,daily_belo_horizonte,daily_belo_horizonte_phi,OOS,730,0.0,92.8,4.217033,11.165378,0.708219,0.0,0.0,0.000,0.00,0.575,15.00,29.255,51.678
2,daily_belo_horizonte,daily_belo_horizonte_phi,Full,4380,0.0,171.8,4.085989,11.614065,0.718037,0.0,0.0,0.000,0.00,0.600,14.20,26.505,53.321
6,monthly_belo_horizonte,monthly_belo_horizonte_phi,IS,216,0.0,934.7,130.506589,148.714573,0.092593,0.0,0.0,14.300,82.15,212.425,332.25,399.325,595.720
7,monthly_belo_horizonte,monthly_belo_horizonte_phi,OOS,24,0.0,410.1,126.504167,135.758806,0.208333,0.0,0.0,4.675,91.70,234.025,328.55,345.585,395.265
8,monthly_belo_horizonte,monthly_belo_horizonte_phi,Full,240,0.0,934.7,130.106347,147.208213,0.104167,0.0,0.0,11.975,82.15,214.050,334.93,397.145,586.792


## OOS forecast cache

Each model has an `oos_forecasts.csv` with expected value, requested quantiles, simulated min/max, and observed value.

In [8]:
for r in all_results:
    print(r['model_id'], r['csv']['oos_forecasts'])
    display(r['oos_forecasts'].head())

daily_belo_horizonte_phi artifacts\cache\daily_belo_horizonte_phi\oos_forecasts.csv


,model_id,sample,t_index,expected_value,q50,q75,q90,q95,q97,q99,sim_min,sim_max,observed
0,daily_belo_horizonte_phi,OOS,0,18.403959,4.740129,14.052235,28.467933,40.508704,49.917950,71.566368,0.0,122.028132,9.9
1,daily_belo_horizonte_phi,OOS,1,16.800394,2.832094,12.497417,27.983409,41.034818,51.265385,74.858838,0.0,136.946281,8.9
2,daily_belo_horizonte_phi,OOS,2,13.456411,1.286761,9.591490,23.389247,35.104535,44.309686,65.575642,0.0,108.399934,21.6
3,daily_belo_horizonte_phi,OOS,3,22.000740,5.680948,16.799643,34.008007,48.380370,59.611359,85.450675,0.0,167.662539,35.4
4,daily_belo_horizonte_phi,OOS,4,28.631487,8.612534,21.830600,41.980652,58.730795,71.797342,101.819243,0.0,156.927913,23.1


daily_belo_horizonte_phi_xi artifacts\cache\daily_belo_horizonte_phi_xi\oos_forecasts.csv


,model_id,sample,t_index,expected_value,q50,q75,q90,q95,q97,q99,sim_min,sim_max,observed
0,daily_belo_horizonte_phi_xi,OOS,0,22.046843,4.136008,12.408699,25.791449,37.428073,46.777709,69.055110,0.0,124.639480,9.9
1,daily_belo_horizonte_phi_xi,OOS,1,16.393323,1.155577,7.360831,19.565558,30.937042,40.330523,63.237097,0.0,129.642559,8.9
2,daily_belo_horizonte_phi_xi,OOS,2,21.100355,2.970845,13.747993,29.330016,42.291980,52.510175,76.455303,0.0,126.547745,21.6
3,daily_belo_horizonte_phi_xi,OOS,3,33.021667,8.160658,20.709865,39.485711,55.262613,67.748745,97.103822,0.0,196.518701,35.4
4,daily_belo_horizonte_phi_xi,OOS,4,23.112035,4.089447,11.978912,25.389732,37.330593,47.027849,70.354426,0.0,116.293449,23.1


monthly_belo_horizonte_phi artifacts\cache_monthly_output_bfgs_216\monthly_belo_horizonte_phi\oos_forecasts.csv


,model_id,sample,t_index,expected_value,q50,q75,q90,q95,q97,q99,sim_min,sim_max,observed
0,monthly_belo_horizonte_phi,OOS,0,0.897071,191.116774,347.339263,527.434510,650.965587,736.907291,910.497290,0.390390,1208.098312,345.5
1,monthly_belo_horizonte_phi,OOS,1,0.680619,145.002640,263.530993,400.171943,493.896804,559.102048,690.807346,0.135794,923.932778,232.6
2,monthly_belo_horizonte_phi,OOS,2,0.293698,62.569139,113.722923,172.693161,213.141907,241.282417,298.122143,0.102484,427.271983,274.4
3,monthly_belo_horizonte_phi,OOS,3,0.332706,70.856386,128.886539,195.778145,241.658326,273.576898,338.046623,0.000000,462.160617,107.0
4,monthly_belo_horizonte_phi,OOS,4,0.068466,14.544215,26.614639,40.519218,50.053100,56.684775,70.077770,0.000000,108.220534,0.0


monthly_belo_horizonte_phi_xi artifacts\cache_monthly_output_bfgs_216\monthly_belo_horizonte_phi_xi\oos_forecasts.csv


,model_id,sample,t_index,expected_value,q50,q75,q90,q95,q97,q99,sim_min,sim_max,observed
0,monthly_belo_horizonte_phi_xi,OOS,0,885.110351,332.853756,494.774541,680.937993,812.405334,906.312334,1102.827659,25.017328,1462.521011,345.5
1,monthly_belo_horizonte_phi_xi,OOS,1,455.908611,164.984568,252.044033,353.688837,426.120576,478.103099,587.398442,7.094549,795.601305,232.6
2,monthly_belo_horizonte_phi_xi,OOS,2,237.956154,72.156353,125.661448,192.804349,242.629382,279.131586,357.439518,0.897972,557.653147,274.4
3,monthly_belo_horizonte_phi_xi,OOS,3,92.483664,15.683402,42.165948,83.860414,118.436386,145.111602,205.111916,0.000000,341.821686,107.0
4,monthly_belo_horizonte_phi_xi,OOS,4,126.131315,7.148940,41.634858,120.872787,197.354721,260.109884,408.606644,0.000000,970.810684,0.0


## Render PDF and LaTeX report

Produces a structured multi-section PDF (and a `.tex` sidecar) with:

1. **Cover page** — abstract and scope note (Belo Horizonte pilot only).
2. **Model descriptions** — ZA-GAS framework, lag sets, phi-only vs phi-xi, parameter counts.
3. **Per-model diagnostics** (one section per model):
   - In-sample PIT histogram
   - In-sample quantile residual ACF (short-range + seasonal lags)
   - KS uniform-fit test table
   - Estimated parameter table with 95 % CIs
4. **Comparison tables** (daily and monthly separately):
   - Performance metrics (IS and OOS) — best value per metric highlighted in green
   - Kupiec / Christoffersen 95 % coverage tests — rejected cells highlighted in red
5. **Conclusion** — summary, next steps, references.

All pages are numbered. The LaTeX sidecar mirrors the structure with `\cellcolor` highlights.

Report regeneration does not call the optimizer by itself; it only uses the `all_results` list created above. To rebuild the same report after fitting/loading, rerun this cell. To make a monthly-only report, use `RUN_SCOPE = 'monthly'` before rebuilding `all_results`; to make the full daily-plus-monthly report, use `RUN_SCOPE = 'both'`.

In [9]:
report_paths = render_model_report(
    all_results,
    pdf_path=REPORT_DIR / 'zagas_phi_vs_phi_xi_comparison.pdf',
    tex_path=REPORT_DIR / 'zagas_phi_vs_phi_xi_comparison.tex',
    title='ZA-GAS: phi-only vs phi-xi — Belo Horizonte Pilot Study',
)
report_paths

{'pdf': WindowsPath('artifacts/reports/zagas_phi_vs_phi_xi_comparison.pdf'),
 'tex': WindowsPath('artifacts/reports/zagas_phi_vs_phi_xi_comparison.tex')}